# Copyright (C) 2026 Bruno Proença de Souza
# Licenciado sob GNU AGPL v3 - veja o arquivo LICENSE

In [7]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.spatial import cKDTree
from shapely.geometry import Point
import warnings

# 🚩 CONFIGURAÇÃO
TIPO_ESCALA = 'quantis'  # 'quantis', 'log' ou 'linear'
ARQUIVO_PARQUET = r'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'

COL_ANO  = 'Ano'
COL_IBGE = 'cod_ibge'
COL_LAT  = 'latitude'
COL_LON  = 'longitude'

colunas_numericas = [
    'Área plantada ou destinada à colheita (Hectares)',
    'Área plantada ou destinada à colheita - percentual do total geral',
    'Área colhida (Hectares)',
    'Área colhida - percentual do total geral',
    'Perda (Hectares)',
    'Quantidade produzida (Toneladas)',
    'Rendimento médio da produção (Quilogramas por Hectare)',
    'Valor da produção (Mil Reais)',
    'Valor da produção - percentual do total geral',
]

# 🚩 0. Carregar parquet
print(f"Carregando '{ARQUIVO_PARQUET}'...")
try:
    df = pd.read_parquet(ARQUIVO_PARQUET)
    print(f"  ✅ {len(df)} linhas | Colunas: {list(df.columns)}")
except Exception as e:
    print(f"  ❌ Erro: {e}")
    raise

# Validar colunas essenciais
for col in [COL_ANO, COL_IBGE, COL_LAT, COL_LON]:
    if col not in df.columns:
        raise ValueError(f"Coluna '{col}' não encontrada no parquet. Colunas disponíveis: {list(df.columns)}")

df[COL_LAT] = pd.to_numeric(df[COL_LAT], errors='coerce')
df[COL_LON] = pd.to_numeric(df[COL_LON], errors='coerce')
df[COL_ANO] = df[COL_ANO].astype(int)

# 🚩 1. Carregar shapefile do Paraná
print("Carregando shapefile do Paraná (geobr)...")
try:
    from geobr import read_state
    estado = read_state(code_state="PR")
except Exception as e:
    print(f"Erro ao carregar shapefile: {e}")
    raise

# 🚩 2. Criar grade de interpolação
print("Criando grade para interpolação...")
minx, miny, maxx, maxy = estado.total_bounds
grid_x, grid_y = np.meshgrid(
    np.linspace(minx, maxx, 300),
    np.linspace(miny, maxy, 300)
)
grid_points = np.vstack([grid_x.ravel(), grid_y.ravel()]).T

grid_points_geom = gpd.GeoSeries([Point(xy) for xy in grid_points], crs=estado.crs)
mask = grid_points_geom.within(estado.union_all())
grid_points = grid_points[mask.values]
print(f"Grade criada com {len(grid_points)} pontos.")

# 🚩 3. Função de interpolação IDW
def idw_interpolation(xy, values, grid_points, k=8, p=2):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        tree = cKDTree(xy)
        distances, indices = tree.query(grid_points, k=k)
        distances = np.where(distances == 0, 1e-10, distances)
        weights = 1.0 / (distances ** p)
        weights /= weights.sum(axis=1)[:, None]
        return (weights * values[indices]).sum(axis=1)

# 🚩 4. Processar dados por ano (CACHE)
print("\nProcessando dados por ano...")
anos_unicos = sorted(df[COL_ANO].unique())
gdf_cache = {}
poligono_pr = estado.union_all()

for ano in anos_unicos:
    print(f"  -> Processando {ano}...")
    df_ano = (
        df[df[COL_ANO] == ano]
        .copy()
        .drop_duplicates(subset=COL_IBGE, keep='first')
        .dropna(subset=[COL_LAT, COL_LON])
    )

    # Calcular coluna Perda
    if 'Área plantada ou destinada à colheita (Hectares)' in df_ano.columns and 'Área colhida (Hectares)' in df_ano.columns:
        df_ano['Perda (Hectares)'] = (
            df_ano['Área plantada ou destinada à colheita (Hectares)'].astype(float)
            - df_ano['Área colhida (Hectares)'].astype(float)
        )

    if df_ano.empty:
        print(f"     ⚠️  Sem dados para {ano}, pulando.")
        continue

    gdf_pontos = gpd.GeoDataFrame(
        df_ano,
        geometry=gpd.points_from_xy(df_ano[COL_LON], df_ano[COL_LAT]),
        crs=estado.crs
    )
    gdf_pontos = gdf_pontos[gdf_pontos.geometry.within(poligono_pr)]

    if not gdf_pontos.empty:
        gdf_cache[ano] = gdf_pontos

print(f"Cache criado para {len(gdf_cache)} anos.")

# 🚩 5. LOOP PRINCIPAL
print("\n" + "="*70)
print("INICIANDO PROCESSAMENTO DE TODAS AS VARIÁVEIS")
print("="*70)

for coluna in colunas_numericas:
    print(f"\n{'='*70}")
    print(f"📊 Processando: {coluna}")
    print(f"{'='*70}")

    todos_valores = []
    dados_interpolados_por_ano = {}
    anos_com_dados = []

    for ano in anos_unicos:
        if ano not in gdf_cache:
            continue

        gdf_pontos = gdf_cache[ano]

        if coluna not in gdf_pontos.columns:
            continue

        dados_validos = gdf_pontos[[coluna, 'geometry']].copy()
        dados_validos[coluna] = pd.to_numeric(dados_validos[coluna], errors='coerce')
        dados_validos = dados_validos.dropna(subset=[coluna])
        dados_validos = dados_validos[np.isfinite(dados_validos[coluna])]

        if len(dados_validos) < 3:
            continue

        valores = dados_validos[coluna].values.astype(float)
        todos_valores.extend(valores)

        xy = np.vstack((dados_validos.geometry.x, dados_validos.geometry.y)).T
        dados_interpolados_por_ano[ano] = idw_interpolation(xy, valores, grid_points)
        anos_com_dados.append(ano)

    if not dados_interpolados_por_ano:
        print(f"  ⚠️  Nenhum dado disponível para '{coluna}'. Pulando...")
        continue

    # 🚩 6. Escala de cores
    todos_valores_array = np.array(todos_valores)
    todos_valores_positivos = todos_valores_array[todos_valores_array > 0]

    if len(todos_valores_positivos) == 0:
        print(f"  ⚠️  Sem valores positivos para '{coluna}'. Pulando...")
        continue

    if TIPO_ESCALA == 'quantis':
        limites = np.unique(np.percentile(todos_valores_positivos, [0, 20, 40, 60, 80, 100]))
    elif TIPO_ESCALA == 'log':
        limites = np.logspace(
            np.log10(np.nanmin(todos_valores_positivos)),
            np.log10(np.nanmax(todos_valores_positivos)), 6
        )
    else:
        limites = np.linspace(np.nanmin(todos_valores_positivos), np.nanmax(todos_valores_positivos), 6)

    print(f"  📈 Limites ({TIPO_ESCALA}): {limites}")

    cores = ['#1a9850', '#91cf60', '#fee090', '#fc8d59', '#d73027'] if coluna == 'Perda (Hectares)' \
            else ['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9850']
    cmap = ListedColormap(cores)
    norm = BoundaryNorm(limites, cmap.N)

    # 🚩 7. Figura
    print(f"  🗺️  Gerando mapa...")
    fig, axes = plt.subplots(3, 3, figsize=(22, 18), squeeze=False)
    axes_flat = axes.flatten()
    sc_ref = None

    for i, ano in enumerate(anos_unicos):
        ax = axes_flat[i]
        if ano not in dados_interpolados_por_ano:
            ax.set_visible(False)
            continue

        sc = ax.scatter(
            grid_points[:, 0], grid_points[:, 1],
            c=dados_interpolados_por_ano[ano],
            cmap=cmap, norm=norm, alpha=0.7, s=8
        )
        if sc_ref is None:
            sc_ref = sc

        estado.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1.2)
        ax.set_title(f'Ano {ano}', fontsize=26, fontweight='bold', pad=10)
        ax.set_xlabel('Longitude', fontsize=26)
        ax.set_ylabel('Latitude', fontsize=26)
        ax.tick_params(axis='x', labelsize=21, rotation=45, pad=3)
        ax.tick_params(axis='y', labelsize=21, pad=3)
        ax.set_aspect('equal', adjustable='box')

    for j in range(len(anos_com_dados), len(axes_flat)):
        axes_flat[j].set_visible(False)

    escala_texto = {'quantis': 'Quantis', 'log': 'Logarítmica', 'linear': 'Linear'}
    fig.suptitle(f'Análise Temporal IDW - {coluna}\n(Escala: {escala_texto[TIPO_ESCALA]})',
                 fontsize=32, fontweight='bold', y=0.98)
    fig.subplots_adjust(top=0.93, bottom=0.06, left=0.06, right=0.88, hspace=0.25, wspace=0.25)

    if sc_ref:
        cbar_ax = fig.add_axes([0.91, 0.15, 0.025, 0.7])
        fmt = '%.0e' if TIPO_ESCALA == 'log' else '%.0f'
        cbar = fig.colorbar(sc_ref, cax=cbar_ax, boundaries=limites, ticks=limites, format=fmt)
        cbar.set_label(coluna, rotation=270, labelpad=70, fontsize=32, fontweight='bold')
        cbar.ax.tick_params(labelsize=28)

    coluna_safe = coluna.replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
    sufixo = f'_{TIPO_ESCALA}' if TIPO_ESCALA != 'linear' else ''
    output_filename = f'mapa_idw_{coluna_safe}{sufixo}.png'
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"  ✅ SALVO: '{output_filename}'")
    plt.close(fig)

print("\n" + "="*70)
print("🎉 PROCESSO CONCLUÍDO PARA TODAS AS VARIÁVEIS!")
print("="*70)

Carregando 'C:\Users\bruno\Desktop\Pipeline_TCC\data\processed\dataset_final.parquet'...
  ✅ 2793 linhas | Colunas: ['cod_ibge', 'Município', 'Ano', 'cod_meso', 'mesorregiao', 'latitude', 'longitude', 'Área plantada ou destinada à colheita (Hectares)', 'Área plantada ou destinada à colheita - percentual do total geral', 'Área colhida (Hectares)', 'Área colhida - percentual do total geral', 'Quantidade produzida (Toneladas)', 'Rendimento médio da produção (Quilogramas por Hectare)', 'Valor da produção (Mil Reais)', 'Valor da produção - percentual do total geral', 'T10M_RANGE_dec1_ano1', 'T10M_RANGE_dec2_ano1', 'T10M_RANGE_dec3_ano1', 'T10M_RANGE_dec4_ano1', 'T10M_RANGE_dec5_ano1', 'T10M_RANGE_dec6_ano1', 'T10M_RANGE_dec7_ano1', 'T10M_RANGE_dec8_ano1', 'T10M_RANGE_dec9_ano1', 'T10M_RANGE_dec10_ano1', 'T10M_RANGE_dec11_ano1', 'T10M_RANGE_dec12_ano1', 'T10M_RANGE_dec13_ano1', 'T10M_RANGE_dec14_ano1', 'T10M_RANGE_dec15_ano1', 'T10M_RANGE_dec16_ano1', 'T10M_RANGE_dec17_ano1', 'T10M_RANGE_dec